In [0]:
import os
import pytest
import pandas as pd
from pyspark.testing.utils import assertSchemaEqual, assertDataFrameEqual
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
SCHEMA_DB = "training_bs.user_uenishi"
SCHEMA_SAS = "training_bs.user_uenishi_sas"
YYYYMM = os.environ.get("P_BASE_YM", "200504")

EXCEL_DB = f"/Volumes/training_bs/user_uenishi/output/uriage_jisseki_{YYYYMM}.xlsx"
EXCEL_SAS = f"/Volumes/training_bs/common/result/uriage_jisseki_{YYYYMM}.xlsx"

tables = [row.name for row in spark.catalog.listTables(SCHEMA_DB)]

In [0]:
class TestTableComparison:
    """DB テーブルと SAS テーブルの比較テスト"""

    @pytest.mark.parametrize("table", tables)
    def test_schema_equal(self, table):
        """スキーマのカラム名・データ型が一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assertSchemaEqual(sdf_db.schema, sdf_sas.schema)

    @pytest.mark.parametrize("table", tables)
    def test_count_equal(self, table):
        """レコード件数が一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assert sdf_db.count() == sdf_sas.count(), \
            f"テーブル {table}: DB={sdf_db.count()}, SAS={sdf_sas.count()}"

    @pytest.mark.parametrize("table", tables)
    def test_data_equal(self, table):
        """全レコードの値が完全一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assertDataFrameEqual(sdf_db, sdf_sas)

In [0]:
class TestExcelComparison:
    """Excel 出力の比較テスト"""

    def test_excel_schema_equal(self):
        """Excel スキーマのカラム名・データ型が一致すること"""
        sdf_db = spark.createDataFrame(pd.read_excel(EXCEL_DB))
        sdf_sas = spark.createDataFrame(pd.read_excel(EXCEL_SAS))
        assertSchemaEqual(sdf_db.schema, sdf_sas.schema)

    def test_excel_count_equal(self):
        """Excel のレコード件数が一致すること"""
        df_db = pd.read_excel(EXCEL_DB)
        df_sas = pd.read_excel(EXCEL_SAS)
        assert len(df_db) == len(df_sas), \
            f"Excel: DB={len(df_db)}, SAS={len(df_sas)}"

    def test_excel_data_equal(self):
        """Excel の全レコードの値が完全一致すること"""
        sdf_db = spark.createDataFrame(pd.read_excel(EXCEL_DB))
        sdf_sas = spark.createDataFrame(pd.read_excel(EXCEL_SAS))
        assertDataFrameEqual(sdf_db, sdf_sas)